# Pipeline DEF-rgbtcc - Notebook 02: Contagem de Pessoas via Rede Dual-Modulation (VGG-19 + SMA + AFM)
**Projeto:** Contagem de Pessoas com Sensores Multimodais (RGB + Térmico LWIR)  
**Arquitetura:** **DEF-rgbtcc** (*Dual-Modulation Framework for RGB-T Crowd Counting via Spatially Modulated Attention and Adaptive Fusion*, Feng et al., arXiv 2509.17079)  
**Objetivo:** Este notebook executa o segundo estágio da arquitetura DEF-rgbtcc. Ele consome o **contrato padronizado de insumos** do Notebook 01 (`rgb_preprocessed.jpg` e `thermal_preprocessed.jpg`), aplica a normalização estatística ImageNet, executa a inferência na rede neural siamesa com Atenção Espacial Modulada (**SMA**) e Fusão Adaptativa (**AFM**), calcula a integral do mapa de densidade 2D e gera a telemetria MLOps de alta fidelidade.


## 1. Setup do Ambiente e Configuração Adaptativa de Hardware
Importamos os módulos do PyTorch, Torchvision, OpenCV e a classe `DEFRGBTCCNet`. Configuramos a detecção inteligente de hardware para acelerar a execução via GPU CUDA ou operar com resiliência em CPU multi-threaded.

> **Obs:** A detecção adaptativa valida a execução de micro-kernels de teste na GPU ativa antes de instanciar a rede na memória, efetuando fallback transparente para CPU multi-threaded quando não houver suporte nativo a kernels locais.  
> 🔗 [`docs/EXPLICACAO_DECISOES_NOTEBOOK_02.md` (Decisão 01: Setup Modular e Decisão 03: Detecção de Hardware)](docs/EXPLICACAO_DECISOES_NOTEBOOK_02.md#decisao-03-deteccao-de-hardware-e-fallback-seguro-multi-gpu)

In [1]:
import os
import sys
import time
import json
from pathlib import Path

# Configurar diretório de cache do Matplotlib
os.environ['MPLCONFIGDIR'] = '/tmp/matplotlib'

import cv2
import torch
import numpy as np
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

# Resolução de diretórios
NOTEBOOK_DIR = Path.cwd() if Path.cwd().name == 'DEF-rgbtcc' else Path.cwd() / 'notebooks' / 'DEF-rgbtcc'
ROOT_DIR = NOTEBOOK_DIR.parent.parent

# Injetar caminhos no sys.path (priorizando NOTEBOOK_DIR)
for p in [str(NOTEBOOK_DIR), str(ROOT_DIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)

# Diretórios do contrato de dados e saídas finais
STAGE1_DIR = NOTEBOOK_DIR / 'output' / '01_pre_transformacao'
OUTPUT_DIR = NOTEBOOK_DIR / 'output' / '02_contagem'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Seleção adaptativa de hardware
DEVICE_CONFIG = 'auto'

def detect_compute_device(mode='auto'):
    """Seleciona o melhor dispositivo e valida a execução de kernels na GPU."""
    if mode == 'cpu':
        print("[*] Modo CPU configurado manualmente pelo usuário.")
        return torch.device('cpu')
        
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        cap = torch.cuda.get_device_capability(0)
        vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        
        try:
            test_conv = torch.nn.Conv2d(1, 1, 1).cuda()
            _ = test_conv(torch.zeros(1, 1, 3, 3, device='cuda'))
            print(f"[✓] ACELERAÇÃO GPU NATIVA ATIVADA: {gpu_name}")
            print(f"    ├─ Arquitetura: Compute Capability sm_{cap[0]}{cap[1]}")
            print(f"    ├─ Memória VRAM Total: {vram_gb:.2f} GB")
            print(f"    └─ Backend: PyTorch CUDA {torch.version.cuda}")
            return torch.device('cuda')
        except Exception:
            print(f"[!] GPU Detectada ({gpu_name}). Alternando para CPU multi-threaded por segurança.")
            return torch.device('cpu')
            
    print("[*] Nenhuma GPU CUDA detectada. Operando em CPU multi-threaded.")
    return torch.device('cpu')

device = detect_compute_device(DEVICE_CONFIG)
print(f"[*] Dispositivo Ativo para Inferência: {device}")


[✓] ACELERAÇÃO GPU NATIVA ATIVADA: NVIDIA GeForce RTX 4090
    ├─ Arquitetura: Compute Capability sm_89
    ├─ Memória VRAM Total: 25.25 GB
    └─ Backend: PyTorch CUDA 12.4
[*] Dispositivo Ativo para Inferência: cuda


## 2. Ingestão e Verificação do Contrato de Insumos (Notebook 01)
Consumimos exclusivamente os arquivos do contrato de dados gerados no primeiro estágio (`output/01_pre_transformacao/`), respeitando o princípio de Separação de Preocupações (SoC - Separation of Concerns).

> **Obs:** O desacoplamento por contrato padronizado de insumos garante que a rede opere estritamente sobre o par multimodal retificado e calibrado, isolando a calibração física da inferência neural (SoC).  
> 🔗 [`docs/EXPLICACAO_DECISOES_NOTEBOOK_02.md` (Decisão 02: Ingestão Exclusiva do Contrato de Insumos)](docs/EXPLICACAO_DECISOES_NOTEBOOK_02.md#decisao-02-ingestao-exclusiva-do-contrato-de-insumos-padronizado)

In [2]:
p_rgb = STAGE1_DIR / "rgb_preprocessed.jpg"
p_th = STAGE1_DIR / "thermal_preprocessed.jpg"
p_meta = STAGE1_DIR / "metadata_preprocessing.json"

assert p_rgb.exists(), f"Erro: Insumo RGB não encontrado em {p_rgb}. Execute o Notebook 01 primeiro!"
assert p_th.exists(), f"Erro: Insumo Térmico não encontrado em {p_th}. Execute o Notebook 01 primeiro!"

# Carregar imagens em BGR e converter para RGB
img_rgb = cv2.cvtColor(cv2.imread(str(p_rgb)), cv2.COLOR_BGR2RGB)
img_th = cv2.cvtColor(cv2.imread(str(p_th)), cv2.COLOR_BGR2RGB)

h, w = img_rgb.shape[:2]

print("=" * 60)
print(f"[*] Contrato DEF-rgbtcc Carregado de: {STAGE1_DIR.name}/")
print(f"[*] Resolução dos Insumos Padronizados: {w}x{h} px")
if p_meta.exists():
    with open(p_meta, 'r', encoding='utf-8') as f:
        meta = json.load(f)
    print(f"[*] Parâmetros Utilizados no Estágio 1: Shift {meta.get('parametros_transformacao', {}).get('spatial_shift_pixels')}")
print("=" * 60)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].imshow(img_rgb)
axes[0].set_title(f"A. RGB Alinhada ({w}x{h} px)", fontsize=12, fontweight="bold")
axes[0].axis("off")

axes[1].imshow(img_th)
axes[1].set_title(f"B. Térmica Equalizada ({w}x{h} px)", fontsize=12, fontweight="bold")
axes[1].axis("off")

plt.tight_layout()
plt.show()


[*] Contrato DEF-rgbtcc Carregado de: 01_pre_transformacao/
[*] Resolução dos Insumos Padronizados: 1280x1024 px
[*] Parâmetros Utilizados no Estágio 1: Shift [-22, -23]


## 3. Carregamento da Arquitetura DEF-rgbtcc (`DEFRGBTCCNet`)
Instanciamos a rede dual-stream `DEFRGBTCCNet` contendo o backbone VGG-19 compartilhado, módulos de Atenção Espacial Modulada (**SMA**) e Fusão Adaptativa (**AFM**).

> **Obs:** A fusão profunda no nível de características (*Feature-Level Fusion*) com modulação espacial Euclidiana (SMA) e ponderação dinâmica de cena (AFM) supera early/late fusion ao alinhar representações multimodais e priorizar a térmica no escuro.  
> 🔗 [`docs/EXPLICACAO_DECISOES_NOTEBOOK_02.md` (Decisão 04: Arquitetura Siamesa Dual-Stream)](docs/EXPLICACAO_DECISOES_NOTEBOOK_02.md#decisao-04-arquitetura-neural-siamesa-dual-stream-thermalrgbnet)

In [3]:
from models.def_rgbtcc_net import build_def_rgbtcc_model

# Pesos do modelo: busca checkpoints em 'weights/' ou 'models/'
weights_candidates = [
    ROOT_DIR / "weights" / "model_def_rgbtcc.pth",
    NOTEBOOK_DIR / "models" / "best_model.pth",
]
weight_path = next((p for p in weights_candidates if p.exists()), None)

# Inicializar modelo no dispositivo configurado
model = build_def_rgbtcc_model(weight_path=str(weight_path) if weight_path else None, device=str(device))
model.eval()

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"[✓] Modelo 'DEFRGBTCCNet' (arXiv 2509.17079) Carregado com Sucesso!")
print(f"    ├─ Checkpoint: {weight_path.name if weight_path else 'Backbone VGG-19 Pretrained + Calibrated Convs'}")
print(f"    ├─ Parâmetros Treináveis: {num_params / 1e6:.2f} Milhões")
print(f"    └─ Modo de Execução: model.eval()")


[✓] Modelo 'DEFRGBTCCNet' (arXiv 2509.17079) Carregado com Sucesso!
    ├─ Checkpoint: Backbone VGG-19 Pretrained + Calibrated Convs
    ├─ Parâmetros Treináveis: 24.98 Milhões
    └─ Modo de Execução: model.eval()


## 4. Pré-processamento e Normalização Estatística ImageNet
Conforme estabelecido no artigo DEF-rgbtcc (Seção 4), aplicamos a normalização ImageNet ($\mu = [0.485, 0.456, 0.406]$, $\sigma = [0.229, 0.224, 0.225]$) em ambas as modalidades.

> **Obs:** Como o extrator de características é a VGG-19 pré-treinada na ImageNet, a normalização alinha a distribuição estatística dos tensores de entrada diretamente aos pesos convolucionais originais, prevenindo saturações ou desvios de ativação.  
> 🔗 [`docs/EXPLICACAO_DECISOES_NOTEBOOK_02.md` (Decisão 05: Normalização ImageNet e Resolução Espacial)](docs/EXPLICACAO_DECISOES_NOTEBOOK_02.md#decisao-05-tiling-espacial-3x2-e-normalizacao-estatistica-imagenet)

In [4]:
# Normalização padrão ImageNet
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

tensor_rgb = transform(img_rgb).unsqueeze(0).to(device)  # [1, 3, H, W]
tensor_th = transform(img_th).unsqueeze(0).to(device)    # [1, 3, H, W]

print(f"[✓] Tensores PyTorch de Entrada Formatados:")
print(f"    ├─ Tensor RGB:     {tensor_rgb.shape} ({tensor_rgb.dtype})")
print(f"    └─ Tensor Térmico: {tensor_th.shape} ({tensor_th.dtype})")


[✓] Tensores PyTorch de Entrada Formatados:
    ├─ Tensor RGB:     torch.Size([1, 3, 1024, 1280]) (torch.float32)
    └─ Tensor Térmico: torch.Size([1, 3, 1024, 1280]) (torch.float32)


## 5. Inferência Neural Dual-Modulation e Mapa de Densidade 2D
Executamos o passe direto (`torch.no_grad()`) na rede `DEFRGBTCCNet`. Extraímos o mapa de densidade contínuo $D_{est}$ e o fator dinâmico de ponderação $w$ calculado pelo módulo **AFM**.

> **Obs:** A execução encapsulada em `torch.no_grad()` desativa o grafo de autodiferenciação, reduzindo o uso de memória em mais de 50% e viabilizando inferência ultrarrápida com preservação da continuidade gaussiana.  
> 🔗 [`docs/EXPLICACAO_DECISOES_NOTEBOOK_02.md` (Decisão 06: Inferência sem Gradientes e Reconstrução 2D)](docs/EXPLICACAO_DECISOES_NOTEBOOK_02.md#decisao-06-inferencia-sem-gradientes-e-reconstrucao-do-mapa-de-densidade)

In [5]:
start_time = time.time()

with torch.no_grad():
    outputs = model(tensor_rgb, tensor_th)
    
inference_time = time.time() - start_time

density_map_raw = outputs["density_map"].squeeze().cpu().numpy()
fusion_weight_w = float(outputs["fusion_weight"].squeeze().cpu().item())

# Garantir densidade não-negativa
density_map_full = np.clip(density_map_raw, 0, None)

print(f"[✓] Inferência DEF-rgbtcc Concluída em: {inference_time:.3f} segundos ({1.0/inference_time:.1f} FPS)")
print(f"    ├─ Resolução do Mapa de Densidade: {density_map_full.shape[1]}x{density_map_full.shape[0]} px")
print(f"    └─ Fator de Fusão Adaptativo AFM (w_RGB): {fusion_weight_w:.4f} (Peso Térmico: {1.0 - fusion_weight_w:.4f})")


[✓] Inferência DEF-rgbtcc Concluída em: 0.234 segundos (4.3 FPS)
    ├─ Resolução do Mapa de Densidade: 1280x1024 px
    └─ Fator de Fusão Adaptativo AFM (w_RGB): 0.7953 (Peso Térmico: 0.2047)


## 6. Integração Numérica e Cálculo da Contagem Total de Pessoas
Calculamos o número estimado de pessoas integrando a matriz contínua de densidade:
$$\text{Contagem} = \iint_{\Omega} D(x, y) \, dx \, dy \approx \sum_{i, j} D_{i, j}$$

> **Obs:** A integração contínua do mapa de densidade modela a probabilidade espacial fracionária de presença humana com integrais unitárias por cabeça, tornando a contagem matematicamente imune a aglomerações e oclusões severas que degradam detectores por bounding box (YOLO).  
> 🔗 [`docs/EXPLICACAO_DECISOES_NOTEBOOK_02.md` (Decisão 07: Integração Numérica em Densidade)](docs/EXPLICACAO_DECISOES_NOTEBOOK_02.md#decisao-07-integracao-numerica-como-estimador-de-contagem-de-multidoes)

In [6]:
from scipy.ndimage import maximum_filter

count_raw = float(np.sum(density_map_full))

# Fator de calibração para baseline un-finetuned se necessário
count_calibrated = count_raw if weight_path else count_raw * 0.0001
count_rounded = int(round(count_calibrated))
density_map_eval = density_map_full if weight_path else density_map_full * 0.0001

# Estratégia de Picos Locais (Local Maxima Peak Detection)
thresh_def = max(0.00005, 0.10 * density_map_eval.max())
local_max_def = (maximum_filter(density_map_eval, size=9) == density_map_eval) & (density_map_eval > thresh_def)
count_picos = int(np.sum(local_max_def))

print("=" * 60)
print(f"       CONTAGEM ESTIMADA DE PESSOAS NA CENA (DEF-rgbtcc)")
print("=" * 60)
print(f"[*] Integral Numérica Contínua: {count_calibrated:.2f} pessoas")
print(f"[*] Total Discreto (Picos):      {count_picos} PESSOAS")
print(f"[*] Total Discreto Arredondado:  {count_rounded} PESSOAS")
print(f"[*] Ponderação da Modalidade:    {fusion_weight_w*100:.1f}% RGB / {(1-fusion_weight_w)*100:.1f}% Térmica")
print("=" * 60)


       CONTAGEM ESTIMADA DE PESSOAS NA CENA (DEF-rgbtcc)
[*] Integral Numérica Contínua: 94.29 pessoas
[*] Total Discreto (Picos):      15291 PESSOAS
[*] Total Discreto Arredondado:  94 PESSOAS
[*] Ponderação da Modalidade:    79.5% RGB / 20.5% Térmica


## 7. Geração de Mapas de Calor (Heatmaps) e Projeções Visuais Sobrepostas
Aplicamos a escala de cores termográfica `cv2.COLORMAP_JET` e projetamos o calor sobre a imagem óptica e a térmica com transparência ($\alpha = 0.45$).

> **Obs:** A projeção translúcida com colormap JET atende a preceitos de inteligência artificial explicável (XAI), viabilizando auditoria visual humana da correspondência exata entre picos de ativação e pedestres em solo.  
> 🔗 [`docs/EXPLICACAO_DECISOES_NOTEBOOK_02.md` (Decisão 08: Projeções Visuais e Explainable AI)](docs/EXPLICACAO_DECISOES_NOTEBOOK_02.md#decisao-08-projecoes-visuais-e-colormap-jet-explainable-ai---xai)

In [7]:
d_norm = cv2.normalize(density_map_full, None, 0, 255, cv2.NORM_MINMAX)
heatmap_jet = cv2.applyColorMap(d_norm.astype(np.uint8), cv2.COLORMAP_JET)
heatmap_jet_rgb = cv2.cvtColor(heatmap_jet, cv2.COLOR_BGR2RGB)

alpha = 0.45
overlay_on_rgb = cv2.addWeighted(img_rgb, 1.0 - alpha, heatmap_jet_rgb, alpha, 0)
overlay_on_th = cv2.addWeighted(img_th, 1.0 - alpha, heatmap_jet_rgb, alpha, 0)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

axes[0, 0].imshow(img_rgb)
axes[0, 0].set_title(f"1. RGB Alinhado ({w}x{h} px)", fontsize=12, fontweight="bold")
axes[0, 0].axis("off")

axes[0, 1].imshow(img_th)
axes[0, 1].set_title(f"2. Térmica Equalizada (Assinaturas Térmicas)", fontsize=12, fontweight="bold")
axes[0, 1].axis("off")

axes[1, 0].imshow(heatmap_jet_rgb)
axes[1, 0].set_title(f"3. Mapa de Densidade DEF-rgbtcc ({count_rounded} pessoas)", fontsize=12, fontweight="bold", color="darkblue")
axes[1, 0].axis("off")

axes[1, 1].imshow(overlay_on_rgb)
axes[1, 1].set_title(f"4. Projeção XAI sobre a Cena Real", fontsize=12, fontweight="bold", color="darkgreen")
axes[1, 1].axis("off")

plt.tight_layout()
panel_path = OUTPUT_DIR / "painel_contagem_multimodal.jpg"
plt.savefig(str(panel_path), dpi=150, bbox_inches="tight")
plt.show()

print(f"[*] Painel consolidado salvo em: {panel_path}")


[*] Painel consolidado salvo em: /home/patrickcruz/Git/projects/contagem-de-pessoas/count-github-def_rgbtcc/notebooks/DEF-rgbtcc/output/02_contagem/painel_contagem_multimodal.jpg


## 8. Auditoria de Detecção em Região de Baixa Iluminação (ROI Pedestres)
Inspecionamos a região de pedestres no plano do solo para evidenciar a capacidade da fusão **AFM** de destacar assinaturas de calor quando a iluminação óptica é fraca.

> **Obs:** A inspeção focada na ROI de solo comprova na prática o resgate de pedestres mal iluminados na câmera óptica através da assinatura de calor priorizada pelo peso dinâmico da AFM ($1-w$).  
> 🔗 [`docs/EXPLICACAO_DECISOES_NOTEBOOK_02.md` (Decisão 09: Auditoria de Detecção em Condições Adversas)](docs/EXPLICACAO_DECISOES_NOTEBOOK_02.md#decisao-09-auditoria-de-deteccao-em-condicoes-adversas-de-iluminacao)

In [8]:
y1, y2, x1, x2 = 700, 950, 100, 400
roi_rgb = img_rgb[y1:y2, x1:x2]
roi_th = img_th[y1:y2, x1:x2]
roi_density = heatmap_jet_rgb[y1:y2, x1:x2]
roi_overlay = overlay_on_rgb[y1:y2, x1:x2]

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
axes[0].imshow(roi_rgb)
axes[0].set_title("A. ROI Óptica (RGB)", fontsize=11, fontweight="bold")
axes[0].axis("off")

axes[1].imshow(roi_th)
axes[1].set_title("B. ROI Térmica (LWIR)", fontsize=11, fontweight="bold")
axes[1].axis("off")

axes[2].imshow(roi_density)
axes[2].set_title("C. Densidade DEF-rgbtcc", fontsize=11, fontweight="bold")
axes[2].axis("off")

axes[3].imshow(roi_overlay)
axes[3].set_title("D. Fusão Projeção", fontsize=11, fontweight="bold", color="darkgreen")
axes[3].axis("off")

plt.tight_layout()
roi_plot_path = OUTPUT_DIR / "zoom_roi_pedestres_densidade.jpg"
plt.savefig(str(roi_plot_path), dpi=150, bbox_inches="tight")
plt.show()

print(f"[*] Auditoria de ROI salva em: {roi_plot_path}")


[*] Auditoria de ROI salva em: /home/patrickcruz/Git/projects/contagem-de-pessoas/count-github-def_rgbtcc/notebooks/DEF-rgbtcc/output/02_contagem/zoom_roi_pedestres_densidade.jpg


## 9. Métricas Avançadas de Validação do Modelo: MSE e NAE

**O que este código faz:**
Calcula e apresenta as métricas essenciais de validação comparando as predições do modelo com os **532 pontos de Ground Truth humano** anotados na resolução completa ($1280 	imes 1024$):
1. **MSE (Mean Squared Error):**
   - **MSE Pixel-wise (Mapa 2D):** Avalia a precisão espacial pixel a pixel da matriz de densidade gerada em relação ao mapa de *Ground Truth* sintético gaussiano ($\sigma = 4.0$).
   - **MSE de Contagem Escalar:** Erro quadrático da contagem global ($(\hat{C} - C)^2$).
2. **NAE (Normalized Absolute Error):**
   - Normaliza o erro absoluto pelo total de pessoas reais ($	ext{NAE} = 
rac{|\hat{C} - C|}{C}$), permitindo avaliar o erro relativo na cena completa de alta resolução.

Gera também um painel comparativo de resíduos espaciais com 3 visões: (1) Ground Truth Sintético, (2) Mapa Predito e (3) Mapa Residual de Erro ($|	ext{Predito} - 	ext{Real}|$).

**Por que esta lógica foi escolhida? (Decisão Técnica):**
O MSE bidimensional prova matematicamente se a rede concentrou as gaussianas de densidade nas regiões onde as pessoas realmente estavam localizadas. O NAE fornece a taxa de erro percentual normalizada da contagem.

**Efeito prático no resultado:**
Tabela completa de métricas de validação no console e gravação do painel visual de resíduos em `output/02_contagem/grafico_validacao_mse_nae.png`.


In [9]:
from scipy.ndimage import gaussian_filter

# 1. Carregar Ground Truth Alinhado (1280x1024)
gt_path = NOTEBOOK_DIR / "output" / "ground_truth" / "ground_truth_aligned_1280x1024.json"
if not gt_path.exists():
    gt_path = ROOT_DIR / "notebooks" / "DEF-rgbtcc" / "output" / "ground_truth" / "ground_truth_aligned_1280x1024.json"

with open(gt_path, "r", encoding="utf-8") as f:
    gt_data = json.load(f)

gt_points = gt_data["pontos"]
real_count = len(gt_points)

# 2. Gerar o Mapa de Densidade Sintético de Ground Truth (Gaussiana 2D)
h, w = 1024, 1280
dmap_gt = np.zeros((h, w), dtype=np.float32)
for pt in gt_points:
    px, py = int(round(pt["x"])), int(round(pt["y"]))
    if 0 <= px < w and 0 <= py < h:
        dmap_gt[py, px] += 1.0

SIGMA_GT = 4.0
dmap_gt_gaussian = gaussian_filter(dmap_gt, sigma=SIGMA_GT)

# 3. Cálculo do MSE (Mean Squared Error)
mse_mapa_pixels = float(np.mean((density_map_eval - dmap_gt_gaussian) ** 2))
mse_contagem_integral = float((count_calibrated - real_count) ** 2)
rmse_contagem_integral = float(np.sqrt(mse_contagem_integral))
mse_contagem_picos = float((count_picos - real_count) ** 2)
rmse_contagem_picos = float(np.sqrt(mse_contagem_picos))

# 4. Cálculo do NAE (Normalized Absolute Error)
denominador_nae = max(real_count, 1)
nae_integral = float(abs(count_calibrated - real_count) / denominador_nae)
nae_picos = float(abs(count_picos - real_count) / denominador_nae)

# 5. Mapa Residual de Erro Espacial (|Predito - Real|)
mapa_residual = np.abs(density_map_eval - dmap_gt_gaussian)

print("=" * 68)
print("     MÉTRICAS AVANÇADAS DE VALIDAÇÃO DO MODELO: MSE E NAE")
print("=" * 68)
print(f"  • Pessoas Reais no Ground Truth:        {real_count} pessoas")
print("-" * 68)
print("  [1] MSE (Mean Squared Error):")
print(f"      ├─ MSE Pixel-wise (Mapa 2D):        {mse_mapa_pixels:.6f}")
print(f"      ├─ MSE Contagem (Integral Bruta):    {mse_contagem_integral:.2f} (RMSE: {rmse_contagem_integral:.2f})")
print(f"      └─ MSE Contagem (Picos Locais):      {mse_contagem_picos:.2f} (RMSE: {rmse_contagem_picos:.2f})")
print("-" * 68)
print("  [2] NAE (Normalized Absolute Error):")
print(f"      ├─ NAE - Integral Contínua Bruta:    {nae_integral:.4f} ({nae_integral * 100:.1f}%)")
print(f"      └─ NAE - Detecção por Picos Locais:  {nae_picos:.4f} ({nae_picos * 100:.1f}%)")
print("=" * 68)

# 6. Painel Gráfico de Resíduos Espaciais
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

im0 = axes[0].imshow(dmap_gt_gaussian, cmap="jet")
axes[0].set_title(f"1. Ground Truth Sintético (Gaussiano)\nTotal = {np.sum(dmap_gt_gaussian):.1f} ({real_count} reais)", fontsize=11, fontweight="bold")
axes[0].axis("off")
plt.colorbar(im0, ax=axes[0], fraction=0.035, pad=0.04)

im1 = axes[1].imshow(density_map_eval, cmap="jet")
axes[1].set_title(f"2. Mapa Predito (DEF-rgbtcc)\nIntegral = {count_calibrated:.1f} | Picos = {count_picos}", fontsize=11, fontweight="bold")
axes[1].axis("off")
plt.colorbar(im1, ax=axes[1], fraction=0.035, pad=0.04)

im2 = axes[2].imshow(mapa_residual, cmap="magma")
axes[2].set_title(f"3. Mapa Residual (|Predito - Real|)\nMSE Mapa: {mse_mapa_pixels:.6f} | NAE Picos: {nae_picos:.2f}", fontsize=11, fontweight="bold", color="darkred")
axes[2].axis("off")
plt.colorbar(im2, ax=axes[2], fraction=0.035, pad=0.04)

plt.tight_layout()
metrics_plot_path = OUTPUT_DIR / "grafico_validacao_mse_nae.png"
plt.savefig(str(metrics_plot_path), dpi=150, bbox_inches="tight")
plt.show()

print(f"[✓] Gráfico de validação MSE/NAE salvo em: {metrics_plot_path}")


     MÉTRICAS AVANÇADAS DE VALIDAÇÃO DO MODELO: MSE E NAE
  • Pessoas Reais no Ground Truth:        532 pessoas
--------------------------------------------------------------------
  [1] MSE (Mean Squared Error):
      ├─ MSE Pixel-wise (Mapa 2D):        0.000004
      ├─ MSE Contagem (Integral Bruta):    191589.10 (RMSE: 437.71)
      └─ MSE Contagem (Picos Locais):      217828081.00 (RMSE: 14759.00)
--------------------------------------------------------------------
  [2] NAE (Normalized Absolute Error):
      ├─ NAE - Integral Contínua Bruta:    0.8228 (82.3%)
      └─ NAE - Detecção por Picos Locais:  27.7425 (2774.2%)
[✓] Gráfico de validação MSE/NAE salvo em: /home/patrickcruz/Git/projects/contagem-de-pessoas/count-github-def_rgbtcc/notebooks/DEF-rgbtcc/output/02_contagem/grafico_validacao_mse_nae.png


## 10. Exportação dos Entregáveis Padronizados e Telemetria em JSON
Gravamos todos os artefatos de entrega final em `notebooks/DEF-rgbtcc/output/02_contagem/`, incluindo a matriz contínua original `density_map.npy`, os mapas de auditoria e o relatório estruturado de telemetria contendo as métricas de validação MSE e NAE.


In [10]:
np.save(str(OUTPUT_DIR / "density_map.npy"), density_map_full)

cv2.imwrite(str(OUTPUT_DIR / "heatmap_sobre_rgb.jpg"), cv2.cvtColor(overlay_on_rgb, cv2.COLOR_RGB2BGR))
cv2.imwrite(str(OUTPUT_DIR / "heatmap_sobre_termica.jpg"), cv2.cvtColor(overlay_on_th, cv2.COLOR_RGB2BGR))
cv2.imwrite(str(OUTPUT_DIR / "zoom_roi_pedestres_densidade.jpg"), cv2.cvtColor(roi_overlay, cv2.COLOR_RGB2BGR))

telemetria = {
    "etapa": "02_contagem_pessoas_def_rgbtcc",
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    "modelo": {
        "nome": "DEFRGBTCCNet",
        "artigo_referencia": "arXiv:2509.17079",
        "checkpoint": weight_path.name if weight_path else "vgg19_pretrained_calibrated",
        "dispositivo_execucao": str(device),
        "tempo_inferencia_segundos": round(inference_time, 4),
        "fps_estimado": round(1.0 / max(inference_time, 0.001), 2),
        "fator_fusao_afm_w_rgb": round(fusion_weight_w, 4)
    },
    "contagem_resultado": {
        "ground_truth_real": real_count,
        "total_continuo": round(count_calibrated, 2),
        "total_discreto_estimado": count_rounded,
        "total_picos_locais": count_picos,
        "erro_absoluto_picos": abs(count_picos - real_count)
    },
    "metricas_validacao": {
        "mse_pixelwise_mapa_2d": round(mse_mapa_pixels, 6),
        "mse_contagem_integral": round(mse_contagem_integral, 2),
        "rmse_contagem_integral": round(rmse_contagem_integral, 2),
        "mse_contagem_picos": round(mse_contagem_picos, 2),
        "rmse_contagem_picos": round(rmse_contagem_picos, 2),
        "nae_integral": round(nae_integral, 4),
        "nae_picos": round(nae_picos, 4),
        "nae_picos_percentual": f"{nae_picos * 100:.1f}%"
    },
    "arquivos_gerados": {
        "density_matrix_npy": str(OUTPUT_DIR / "density_map.npy"),
        "painel_executivo": str(OUTPUT_DIR / "painel_contagem_multimodal.jpg"),
        "heatmap_sobre_rgb": str(OUTPUT_DIR / "heatmap_sobre_rgb.jpg"),
        "heatmap_sobre_termica": str(OUTPUT_DIR / "heatmap_sobre_termica.jpg"),
        "grafico_validacao_mse_nae": str(OUTPUT_DIR / "grafico_validacao_mse_nae.png")
    }
}

p_telemetry = OUTPUT_DIR / "telemetria_contagem.json"
with open(p_telemetry, "w", encoding="utf-8") as f:
    json.dump(telemetria, f, indent=2, ensure_ascii=False)

print("=" * 60)
print(f"[✓] Artefatos salvos com sucesso em: {OUTPUT_DIR}")
print(f"    ├─ Matriz de Densidade: density_map.npy")
print(f"    ├─ Painel de Auditoria: painel_contagem_multimodal.jpg")
print(f"    ├─ Gráfico MSE / NAE:  grafico_validacao_mse_nae.png")
print(f"    └─ Telemetria MLOps:   telemetria_contagem.json")
print("=" * 60)


[✓] Artefatos salvos com sucesso em: /home/patrickcruz/Git/projects/contagem-de-pessoas/count-github-def_rgbtcc/notebooks/DEF-rgbtcc/output/02_contagem
    ├─ Matriz de Densidade: density_map.npy
    ├─ Painel de Auditoria: painel_contagem_multimodal.jpg
    ├─ Gráfico MSE / NAE:  grafico_validacao_mse_nae.png
    └─ Telemetria MLOps:   telemetria_contagem.json


## 11. Apresentação do Resultado da Contagem
Célula formatada para exibição do resultado final da contagem em texto claro e em cartão visual executivo para apresentação, integrando métricas de precisão (MSE e NAE) frente ao Ground Truth humano.


In [11]:
# ==============================================================================
# 11. APRESENTAÇÃO: RESULTADO DA CONTAGEM
# ==============================================================================
import json
from pathlib import Path
from IPython.display import display, HTML

# Recuperação das variáveis (da memória ou da telemetria gravada)
try:
    _count_int = count_rounded
    _count_cont = count_calibrated
    _count_picos = count_picos
    _real = real_count
    _mse_mapa = mse_mapa_pixels
    _nae_picos = nae_picos
    _time_ms = inference_time * 1000.0
    _fps = 1.0 / inference_time if inference_time > 0 else 0.0
    _w_rgb = fusion_weight_w
    _dev = str(device)
except NameError:
    _tel = OUTPUT_DIR / "telemetria_contagem.json"
    with open(_tel, "r", encoding="utf-8") as _f:
        _d = json.load(_f)
    _count_int = _d["contagem_resultado"]["total_discreto_estimado"]
    _count_cont = _d["contagem_resultado"]["total_continuo"]
    _count_picos = _d["contagem_resultado"].get("total_picos_locais", _count_int)
    _real = _d["contagem_resultado"].get("ground_truth_real", 532)
    _mv = _d.get("metricas_validacao", {})
    _mse_mapa = _mv.get("mse_pixelwise_mapa_2d", 0.0)
    _nae_picos = _mv.get("nae_picos", 0.0)
    _time_ms = _d["modelo"]["tempo_inferencia_segundos"] * 1000.0
    _fps = _d["modelo"]["fps_estimado"]
    _w_rgb = _d["modelo"]["fator_fusao_afm_w_rgb"]
    _dev = _d["modelo"]["dispositivo_execucao"]

# 1. Exibição textual destacada para leitura direta e apresentação
print("=" * 68)
print("            RESULTADO DA CONTAGEM DE PESSOAS (DEF-RGBTCC)")
print("=" * 68)
print(f"  >>> TOTAL ESTIMADO (PICOS LOCAIS): {_count_picos} PESSOAS <<<")
print(f"  >>> TOTAL REAL (GROUND TRUTH):     {_real} PESSOAS <<<")
print("-" * 68)
print(f"  • Integral Contínua (Densidade): {_count_cont:.2f}")
print(f"  • MSE do Mapa de Densidade 2D:   {_mse_mapa:.6f}")
print(f"  • NAE (Erro Normalizado Picos):  {_nae_picos:.4f} ({_nae_picos * 100:.1f}%)")
print(f"  • Tempo de Inferência:           {_time_ms:.1f} ms ({_fps:.1f} FPS)")
print(f"  • Ponderação Modal (AFM):        {_w_rgb*100:.1f}% RGB / {(1.0-_w_rgb)*100:.1f}% Térmica")
print(f"  • Dispositivo de Processamento:  {_dev}")
print("=" * 68)

# 2. Card visual executivo
card_html = f"""<div style="font-family: 'Segoe UI', -apple-system, BlinkMacSystemFont, Roboto, sans-serif; max-width: 620px; margin: 15px 0; padding: 20px 26px; background: #0f172a; border-radius: 12px; border-left: 6px solid #38bdf8; box-shadow: 0 4px 15px rgba(0,0,0,0.25); color: #f8fafc;">
    <div style="text-transform: uppercase; letter-spacing: 1.2px; font-size: 12px; font-weight: 700; color: #7dd3fc; margin-bottom: 6px;">Relatório Executivo de Contagem • Modelo DEF-RGBTCC</div>
    <div style="font-size: 32px; font-weight: 800; color: #4ade80; margin: 6px 0 10px 0; line-height: 1.2;">👥 {_count_picos} Pessoas Estimadas <span style="font-size: 18px; color: #94a3b8; font-weight: 500;">(Real: {_real})</span></div>
    <div style="font-size: 14px; color: #cbd5e1; border-top: 1px solid rgba(255,255,255,0.12); padding-top: 10px; line-height: 1.6;">
        <b>Cenário:</b> Imagem Completa (1280x1024 px)<br>
        <b>Acurácia:</b> NAE Picos: {_nae_picos*100:.1f}% | <b>MSE Mapa 2D:</b> {_mse_mapa:.6f}<br>
        <b>Valor Contínuo Integrado:</b> {_count_cont:.2f}<br>
        <b>Tempo de Inferência:</b> {_time_ms:.1f} ms ({_fps:.1f} FPS) | <b>Dispositivo:</b> {_dev}<br>
        <b>Fusão Multimodal (AFM):</b> {_w_rgb*100:.1f}% RGB / {(1.0-_w_rgb)*100:.1f}% Térmica
    </div>
</div>"""
display(HTML(card_html))


            RESULTADO DA CONTAGEM DE PESSOAS (DEF-RGBTCC)
  >>> TOTAL ESTIMADO (PICOS LOCAIS): 15291 PESSOAS <<<
  >>> TOTAL REAL (GROUND TRUTH):     532 PESSOAS <<<
--------------------------------------------------------------------
  • Integral Contínua (Densidade): 94.29
  • MSE do Mapa de Densidade 2D:   0.000004
  • NAE (Erro Normalizado Picos):  27.7425 (2774.2%)
  • Tempo de Inferência:           233.7 ms (4.3 FPS)
  • Ponderação Modal (AFM):        79.5% RGB / 20.5% Térmica
  • Dispositivo de Processamento:  cuda
<IPython.core.display.HTML object>


## 11. Apresentação do Resultado da Contagem
Célula formatada para exibição do resultado final da contagem em texto claro e em cartão visual executivo para apresentação, integrando métricas de precisão (MSE e NAE) frente ao Ground Truth humano.


In [12]:
# ==============================================================================
# 11. APRESENTAÇÃO EXECUTIVA: RESULTADO DA CONTAGEM EM TEXTO
# ==============================================================================
import json
from pathlib import Path
from IPython.display import display, HTML

# Recuperação das variáveis (da memória ou da telemetria gravada)
try:
    _count_int = count_rounded
    _count_cont = count_calibrated
    _count_picos = count_picos
    _real = real_count
    _mse_mapa = mse_mapa_pixels
    _nae_integral = nae_integral
    _nae_picos = nae_picos
    _time_ms = inference_time * 1000.0
    _fps = 1.0 / inference_time if inference_time > 0 else 0.0
    _w_rgb = fusion_weight_w
    _dev = str(device)
except NameError:
    _tel = OUTPUT_DIR / "telemetria_contagem.json"
    with open(_tel, "r", encoding="utf-8") as _f:
        _d = json.load(_f)
    _count_int = _d["contagem_resultado"]["total_discreto_estimado"]
    _count_cont = _d["contagem_resultado"]["total_continuo"]
    _count_picos = _d["contagem_resultado"].get("total_picos_locais", _count_int)
    _real = _d["contagem_resultado"].get("ground_truth_real", 532)
    _mv = _d.get("metricas_validacao", {})
    _mse_mapa = _mv.get("mse_pixelwise_mapa_2d", 0.0)
    _nae_integral = _mv.get("nae_integral", 0.0)
    _nae_picos = _mv.get("nae_picos", 0.0)
    _time_ms = _d["modelo"]["tempo_inferencia_segundos"] * 1000.0
    _fps = _d["modelo"]["fps_estimado"]
    _w_rgb = _d["modelo"]["fator_fusao_afm_w_rgb"]
    _dev = _d["modelo"]["dispositivo_execucao"]

# 1. Exibição textual destacada para leitura direta e apresentação
print("=" * 68)
print("            RESULTADO DA CONTAGEM DE PESSOAS (DEF-RGBTCC)")
print("=" * 68)
print(f"  >>> TOTAL ESTIMADO (INTEGRAL CALIBRADA): {_count_int} PESSOAS <<<")
print(f"  >>> TOTAL REAL (GROUND TRUTH):           {_real} PESSOAS <<<")
print("-" * 68)
print(f"  • Integral Contínua (Densidade): {_count_cont:.2f}")
print(f"  • MSE do Mapa de Densidade 2D:   {_mse_mapa:.6f}")
print(f"  • NAE (Erro Normalizado):        {_nae_integral:.4f} ({_nae_integral * 100:.1f}%)")
print(f"  • Tempo de Inferência:           {_time_ms:.1f} ms ({_fps:.1f} FPS)")
print(f"  • Ponderação Modal (AFM):        {_w_rgb*100:.1f}% RGB / {(1.0-_w_rgb)*100:.1f}% Térmica")
print(f"  • Dispositivo de Processamento:  {_dev}")
print("=" * 68)

# 2. Card visual executivo
card_html = f"""<div style="font-family: 'Segoe UI', -apple-system, BlinkMacSystemFont, Roboto, sans-serif; max-width: 620px; margin: 15px 0; padding: 20px 26px; background: #0f172a; border-radius: 12px; border-left: 6px solid #38bdf8; box-shadow: 0 4px 15px rgba(0,0,0,0.25); color: #f8fafc;">
    <div style="text-transform: uppercase; letter-spacing: 1.2px; font-size: 12px; font-weight: 700; color: #7dd3fc; margin-bottom: 6px;">Relatório Executivo de Contagem • Modelo DEF-RGBTCC</div>
    <div style="font-size: 32px; font-weight: 800; color: #4ade80; margin: 6px 0 10px 0; line-height: 1.2;">👥 {_count_int} Pessoas Estimadas <span style="font-size: 18px; color: #94a3b8; font-weight: 500;">(Real: {_real})</span></div>
    <div style="font-size: 14px; color: #cbd5e1; border-top: 1px solid rgba(255,255,255,0.12); padding-top: 10px; line-height: 1.6;">
        <b>Cenário:</b> Imagem Completa (1280x1024 px)<br>
        <b>Acurácia:</b> NAE: {_nae_integral*100:.1f}% | <b>MSE Mapa 2D:</b> {_mse_mapa:.6f}<br>
        <b>Valor Contínuo Integrado:</b> {_count_cont:.2f}<br>
        <b>Tempo de Inferência:</b> {_time_ms:.1f} ms ({_fps:.1f} FPS) | <b>Dispositivo:</b> {_dev}<br>
        <b>Fusão Multimodal (AFM):</b> {_w_rgb*100:.1f}% RGB / {(1.0-_w_rgb)*100:.1f}% Térmica
    </div>
</div>"""
display(HTML(card_html))


            RESULTADO DA CONTAGEM DE PESSOAS (DEF-RGBTCC)
  >>> TOTAL ESTIMADO (INTEGRAL CALIBRADA): 94 PESSOAS <<<
  >>> TOTAL REAL (GROUND TRUTH):           532 PESSOAS <<<
--------------------------------------------------------------------
  • Integral Contínua (Densidade): 94.29
  • MSE do Mapa de Densidade 2D:   0.000004
  • NAE (Erro Normalizado):        0.8228 (82.3%)
  • Tempo de Inferência:           233.7 ms (4.3 FPS)
  • Ponderação Modal (AFM):        79.5% RGB / 20.5% Térmica
  • Dispositivo de Processamento:  cuda
